In [ ]:
%%capture
%pip install pycox lifelines scikit-survival optuna tqdm

In [ ]:
# Standard library imports
import gc
import os
import random
import warnings

# Third-party imports
import numpy as np
import optuna
import pandas as pd
import torch
import torch.nn as nn
import torchtuples as tt
import xgboost as xgb
from lifelines import CoxPHFitter
from lifelines.utils import concordance_index
from matplotlib import pyplot as plt
from pycox.models import CoxPH
from sklearn.feature_selection import VarianceThreshold
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.utils import resample
from sksurv.datasets import load_breast_cancer
from sksurv.ensemble import RandomSurvivalForest
from sksurv.linear_model import CoxnetSurvivalAnalysis
from sksurv.metrics import concordance_index_censored
from tqdm.auto import tqdm
from pycox.models import DeepHitSingle
from pycox.preprocessing.label_transforms import LabTransDiscreteTime
from sksurv.datasets import load_whas500
from lifelines.datasets import load_rossi
from sksurv.metrics import integrated_brier_score

In [ ]:
# cox proportional hazards
def cph(X_train, T_train, E_train, X_test, times):
    df_train = pd.DataFrame(X_train)
    df_train["time"] = T_train
    df_train["event"] = E_train

    cph = CoxPHFitter(penalizer=0.0)  # no penalisation

    try:
        cph.fit(df_train, duration_col="time", event_col="event")

        # predictions flattened to a 1D array
        preds = cph.predict_partial_hazard(pd.DataFrame(X_test)).values.flatten()

        # return 2d  array for IBS
        surv = cph.predict_survival_function(pd.DataFrame(X_test), times=times).values.T

        return preds, surv

    except Exception as e:
        print(f"CPH failed to converge: {e}")
        return None

In [ ]:
def cox_lasso(X_train, T_train, E_train, X_test, times=None):
    try:
        # scikit-survival requires a structured array for the target variable
        y_train = np.array([(bool(e), t) for e, t in zip(E_train, T_train)],
                           dtype=[('event', 'bool'), ('time', 'float32')])

        # Initialize Lasso (l1_ratio=1.0 is pure Lasso / L1 penalty)
        lasso_model = CoxnetSurvivalAnalysis(l1_ratio=1.0, fit_baseline_model=True)

        # fit model
        lasso_model.fit(X_train, y_train)

        # predict and flatten to ensure a clean 1D array for the bootstrap loop
        risk_scores = lasso_model.predict(X_test).flatten()

        # 2d array of survival functions for IBS
        surv_funcs = lasso_model.predict_survival_function(X_test)
        surv = np.array([fn(times) for fn in surv_funcs])

        return risk_scores, surv

    except Exception as e:
        print(f"Cox Lasso failed to converge: {e}")
        return None

In [ ]:
# random survival forest
def rsf(X_train, T_train, E_train, X_test, times=None):
    try:
        y_train = np.array([(bool(e), t) for e, t in zip(E_train, T_train)], dtype=[("e", bool), ("t", float)])

        # fit rsf
        rsf_model = RandomSurvivalForest(n_estimators=100, min_samples_leaf=15, n_jobs=-1, random_state=0)
        rsf_model.fit(X_train, y_train)

        # risk for c-index
        risk = rsf_model.predict(X_test)

        # survival functions for IBS
        surv_funcs = rsf_model.predict_survival_function(X_test)
        surv = np.array([fn(times) for fn in surv_funcs])

        return risk, surv

    except Exception as e:
        print(f"RSF failed: {e}")
        return None, None if times is not None else None

In [ ]:
def deepsurv(X_train, T_train, E_train, X_test, params, times):
    try:
        # Ensure arrays are float32 for PyTorch compatibility
        X_train = X_train.astype("float32")
        T_train = T_train.astype("float32")
        E_train = E_train.astype("float32")
        X_test = X_test.astype("float32")

        # Split strictly within the training set for Early Stopping
        X_tr, X_val, T_tr, T_val, E_tr, E_val = train_test_split(
            X_train, T_train, E_train, test_size=0.2, random_state=42, stratify=E_train
        )

        y_tr = (T_tr, E_tr)
        y_val = (T_val, E_val)

        in_features = X_tr.shape[1]
        out_features = 1
        num_nodes = [params["nodes"]] * params["layers"]
        dropout = params["dropout"]
        batch_norm = params.get("batch_norm", True)

        activation = nn.SELU if params.get("activation", "ReLU") == "SELU" else nn.ReLU

        net = tt.practical.MLPVanilla(
            in_features, num_nodes, out_features, batch_norm,
            dropout, activation=activation, output_bias=False
        )

        if params.get("optimiser", "adam") == "adam":
            optim = tt.optim.Adam(lr=params["lr"], weight_decay=params.get("l2", 0.0))
        elif params.get("optimiser") == "sgd":
            optim = tt.optim.SGD(lr=params["lr"], momentum=params.get("momentum", 0.9),
                                 weight_decay=params.get("l2", 0.0), nesterov=True)
        else:
            raise ValueError("Unknown optimiser")

        device = "cuda" if torch.cuda.is_available() else "cpu"
        model = CoxPH(net, optim, device=device)

        callbacks = [tt.callbacks.EarlyStopping(patience=15)]
        batch_size = params.get("batch_size", 64)

        model.fit(
            X_tr, y_tr,
            batch_size=batch_size,
            epochs=500,
            callbacks=callbacks,
            verbose=False,
            val_data=(X_val, y_val)
        )

        # Return the flat array of log-hazard (risk) scores
        risk = model.predict_net(X_test).flatten()

        # survival probabilities
        _ = model.compute_baseline_hazards(X_tr, y_tr)  # calculate baseline hazards
        surv_df = model.predict_surv_df(X_test)
        surv = surv_df.reindex(times, method='ffill').bfill().values.T

        return risk, surv

    except Exception as e:
        print(f"DeepSurv failed: {e}")
        return None

In [ ]:
def optimise_deepsurv(X_train, T_train, E_train, n_trials=30):
    X_train, T_train, E_train = X_train.astype("float32"), T_train.astype("float32"), E_train.astype("float32")

    X_tr, X_val, T_tr, T_val, E_tr, E_val = train_test_split(
        X_train, T_train, E_train, test_size=0.2, random_state=42, stratify=E_train
    )

    def objective(trial):
        # parameters to optimise
        n_nodes = trial.suggest_int("nodes", 16, 128, step=16)
        n_layers = trial.suggest_int("layers", 1, 4)
        dropout = trial.suggest_float("dropout", 0.1, 0.6)
        lr = trial.suggest_float("lr", 1e-4, 1e-2, log=True)
        l2 = trial.suggest_float("l2", 1e-4, 1e-1, log=True)
        batch_size = trial.suggest_categorical("batch_size", [32, 64, 128])
        activation_name = trial.suggest_categorical("activation", ["ReLU", "SELU"])
        batch_norm = trial.suggest_categorical("batch_norm", [True, False])

        in_features = X_tr.shape[1]
        out_features = 1
        num_nodes = [n_nodes] * n_layers
        activation = nn.SELU if activation_name == "SELU" else nn.ReLU

        # deepsurv
        net = tt.practical.MLPVanilla(
            in_features, num_nodes, out_features,
            batch_norm=batch_norm, dropout=dropout,
            activation=activation, output_bias=False
        )

        optim = tt.optim.Adam(lr=lr, weight_decay=l2)
        device = "cuda" if torch.cuda.is_available() else "cpu"
        model = CoxPH(net, optim, device=device)

        callbacks = [tt.callbacks.EarlyStopping(patience=15)]

        model.fit(
            X_tr, (T_tr, E_tr),
            batch_size=batch_size, epochs=500,
            callbacks=callbacks, verbose=False,
            val_data=(X_val, (T_val, E_val))
        )

        risk = model.predict_net(X_val).flatten()
        c_index = concordance_index(T_val, -risk, E_val)

        # memory cleanup
        del model, net, optim
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        gc.collect()

        return c_index

    optuna.logging.set_verbosity(optuna.logging.WARNING)
    study = optuna.create_study(direction="maximize")
    study.optimize(objective, n_trials=n_trials, n_jobs=1, show_progress_bar=True)

    return study.best_params

In [ ]:
def deephit(X_train, T_train, E_train, X_test, params, times):
    try:
        # Ensure float32 for PyTorch compatibility
        X_train = X_train.astype("float32")
        T_train = T_train.astype("float32")
        E_train = E_train.astype("float32")
        X_test = X_test.astype("float32")

        # extract parameters
        alpha = params.get("alpha", 0.2)
        sigma = params.get("sigma", 0.1)
        num_durations = params.get("num_durations", 10)

        # discretise time for DeepHit
        labtrans = LabTransDiscreteTime(num_durations)
        # Fit the transformer on training data and transform it
        y_train_discrete = labtrans.fit_transform(T_train, E_train)

        # Split strictly within the training set for Early Stopping
        X_tr, X_val, y_tr_time, y_val_time, y_tr_event, y_val_event = train_test_split(
            X_train, y_train_discrete[0], y_train_discrete[1],
            test_size=0.2, random_state=42, stratify=y_train_discrete[1]
        )

        y_tr = (y_tr_time, y_tr_event)
        y_val = (y_val_time, y_val_event)

        # build the network
        in_features = X_train.shape[1]
        out_features = labtrans.out_features # Network must output exactly the number of time bins
        num_nodes = [params.get("nodes", 32)] * params.get("layers", 2)
        dropout = params.get("dropout", 0.2)
        batch_norm = params.get("batch_norm", True)

        activation = nn.SELU if params.get("activation", "ReLU") == "SELU" else nn.ReLU

        net = tt.practical.MLPVanilla(
            in_features, num_nodes, out_features, batch_norm,
            dropout, activation=activation, output_bias=False
        )

        optim = tt.optim.Adam(lr=params.get("lr", 1e-3), weight_decay=params.get("l2", 0.01))
        device = "cuda" if torch.cuda.is_available() else "cpu"

        # deephit
        model = DeepHitSingle(net, optim, alpha=alpha, sigma=sigma, duration_index=labtrans.cuts, device=device)

        callbacks = [tt.callbacks.EarlyStopping(patience=15)]

        # train
        model.fit(
            X_tr, y_tr,
            batch_size=params.get("batch_size", 64),
            epochs=500,
            callbacks=callbacks,
            verbose=False,
            val_data=(X_val, y_val)
        )

        # predict
        surv_df = model.predict_surv_df(X_test)

        # compute risk
        expected_survival_time = surv_df.sum().values
        risk = -expected_survival_time

        # survival probabilities
        surv = surv_df.reindex(times, method='ffill').bfill().values.T

        return risk, surv

    except Exception as e:
        print(f"DeepHit failed: {e}")
        return None

In [ ]:
def optimise_deephit(X_train, T_train, E_train, n_trials=15):
    X_train = X_train.astype("float32")
    T_train = T_train.astype("float32")
    E_train = E_train.astype("float32")

    def objective(trial):
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

        # parameters to optimise
        n_nodes = trial.suggest_int("nodes", 32, 64, step=32)
        n_layers = trial.suggest_int("layers", 1, 3)
        dropout = trial.suggest_float("dropout", 0.2, 0.6)
        lr = trial.suggest_float("lr", 1e-4, 1e-2, log=True)
        batch_size = trial.suggest_categorical("batch_size", [64, 128])

        alpha = trial.suggest_float("alpha", 0.0, 1.0)
        sigma = trial.suggest_float("sigma", 0.1, 1.0)
        num_durations = trial.suggest_categorical("num_durations", [10, 20])

        # deephit
        labtrans = LabTransDiscreteTime(num_durations)
        y_train_discrete = labtrans.fit_transform(T_train, E_train)

        X_tr, X_val, y_tr_time, y_val_time, y_tr_event, y_val_event, T_tr_orig, T_val_orig, E_tr_orig, E_val_orig = train_test_split(
            X_train, y_train_discrete[0], y_train_discrete[1], T_train, E_train,
            test_size=0.2, random_state=42, stratify=y_train_discrete[1]
        )

        y_tr = (y_tr_time, y_tr_event)
        y_val = (y_val_time, y_val_event)

        # build the network
        in_features = X_tr.shape[1]
        out_features = labtrans.out_features
        num_nodes_list = [n_nodes] * n_layers

        net = tt.practical.MLPVanilla(
            in_features, num_nodes_list, out_features,
            batch_norm=True, dropout=dropout,
            activation=nn.ReLU, output_bias=False
        )

        optim = tt.optim.Adam(lr=lr)
        device = "cuda" if torch.cuda.is_available() else "cpu"

        model = DeepHitSingle(net, optim, alpha=alpha, sigma=sigma, duration_index=labtrans.cuts, device=device)
        callbacks = [tt.callbacks.EarlyStopping(patience=10)] # Lowered patience for speed/memory

        # train the model
        model.fit(
            X_tr, y_tr,
            batch_size=batch_size, epochs=200, # Lowered max epochs
            callbacks=callbacks, verbose=False,
            val_data=(X_val, y_val)
        )

        # compute risk
        surv_df = model.predict_surv_df(X_val)
        expected_survival_time = surv_df.sum().values
        risk = -expected_survival_time

        c_index = concordance_index(T_val_orig, risk, E_val_orig)

        # memory cleanup
        del model, net, optim, surv_df
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

        return c_index

    # Run the study
    optuna.logging.set_verbosity(optuna.logging.WARNING)
    study = optuna.create_study(direction="maximize")
    study.optimize(objective, n_trials=n_trials, n_jobs=1, show_progress_bar=True)

    return study.best_params

In [ ]:
# import tcga dataset from google drive (processed in other notebook)
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

df_tcga = pd.read_csv('/content/drive/MyDrive/tcga_ready_for_ml.csv')

T_tcga = df_tcga["OS_MONTHS"].values.astype("float32")
E_tcga = df_tcga["OS_STATUS"].values.astype("float32")

# Drop the targets and colinear feature
cols_to_drop = ["OS_MONTHS", "OS_STATUS", "PANEL_MUT_BURDEN"]

X_tcga = df_tcga.drop(columns=cols_to_drop).values.astype("float32")
selector = VarianceThreshold(threshold=0.0)
X_tcga = selector.fit_transform(X_tcga)

print(f"Features (X) shape: {X_tcga.shape}")
print(f"Durations (T) shape: {T_tcga.shape}")

In [ ]:
# TCGA
X_train_tcga, X_test_tcga, T_train_tcga, T_test_tcga, E_train_tcga, E_test_tcga = train_test_split(
    X_tcga, T_tcga, E_tcga, test_size=0.2, random_state=42, stratify=E_tcga
)

n_trials = 20

scaler_tcga = StandardScaler()
X_train_tcga_scaled = scaler_tcga.fit_transform(X_train_tcga)
best_tcga_params = optimise_deepsurv(X_train_tcga_scaled, T_train_tcga, E_train_tcga, n_trials=n_trials)
params_tcga = best_tcga_params.copy()

In [ ]:
print(f"params_tcga = {params_tcga}")

In [ ]:
n_trials_dh = 15

# TCGA
scaler_tcga = StandardScaler()
X_train_tcga_scaled = scaler_tcga.fit_transform(X_train_tcga)
best_tcga_params_dh = optimise_deephit(X_train_tcga_scaled, T_train_tcga, E_train_tcga, n_trials=n_trials_dh)
params_tcga_dh = best_tcga_params_dh.copy()

In [ ]:
print(f"params_tcga_dh = {params_tcga_dh}")

In [ ]:
def evaluate_dataset(X_train, T_train, E_train,
                     X_test, T_test, E_test,
                     dataset_name, params_ds, params_dh,
                     n_bootstraps=1000):

    # Scale features strictly on the training data to prevent leakage
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    # format targets for C-index
    T_test = np.array(T_test)
    E_test = np.array(E_test)

    # format targets for IBS
    y_train_sksurv = np.array([(bool(e), t) for e, t in zip(E_train, T_train)], dtype=[('e', bool), ('t', float)])
    y_test_sksurv = np.array([(bool(e), t) for e, t in zip(E_test, T_test)], dtype=[('e', bool), ('t', float)])

    # define the time grid for IBS integration (5th to 95th percentile of test events)
    mask = E_test == 1
    event_times = T_test[mask]
    times = np.linspace(np.percentile(event_times, 5), np.percentile(event_times, 95), 100)

    # train models and get predictions
    risk_cph, surv_cph = cph(X_train_scaled, T_train, E_train, X_test_scaled, times=times)
    risk_lasso, surv_lasso = cox_lasso(X_train_scaled, T_train, E_train, X_test_scaled, times=times)
    risk_ds, surv_ds = deepsurv(X_train_scaled, T_train, E_train, X_test_scaled, params_ds, times=times)
    risk_dh, surv_dh = deephit(X_train_scaled, T_train, E_train, X_test_scaled, params_dh, times=times)
    risk_rsf, surv_rsf = rsf(X_train_scaled, T_train, E_train, X_test_scaled, times=times)

    # Initialise lists for C-index
    cidx_cph, cidx_lasso, cidx_ds, cidx_rsf, cidx_dh = [], [], [], [], []
    # Initialise lists for IBS
    ibs_cph, ibs_lasso, ibs_ds, ibs_rsf, ibs_dh = [], [], [], [], []

    # Bootstrap loop
    for i in tqdm(range(n_bootstraps), desc=f"Bootstrapping {dataset_name}", leave=False):
        # Resample the test set indices with replacement
        indices = resample(np.arange(len(X_test_scaled)), replace=True, random_state=i)

        T_boot = T_test[indices]
        E_boot = E_test[indices]
        y_test_boot = y_test_sksurv[indices]

        if np.sum(E_boot) == 0:
            continue

        # concordance index
        cidx_cph.append(concordance_index(T_boot, -risk_cph[indices], E_boot))
        cidx_lasso.append(concordance_index(T_boot, -risk_lasso[indices], E_boot))
        cidx_ds.append(concordance_index(T_boot, -risk_ds[indices], E_boot))
        cidx_dh.append(concordance_index(T_boot, -risk_dh[indices], E_boot))
        cidx_rsf.append(concordance_index(T_boot, -risk_rsf[indices], E_boot))

        time_mask = (times >= T_boot.min()) & (times < T_boot.max())
        times_boot = times[time_mask]

        # Skip this bootstrap sample for IBS if it restricted the times too heavily (needs >= 2 points to integrate)
        if len(times_boot) < 2:
            continue

        # integrated brier score
        ibs_cph.append(integrated_brier_score(y_train_sksurv, y_test_boot, surv_cph[indices][:, time_mask], times_boot))
        ibs_lasso.append(integrated_brier_score(y_train_sksurv, y_test_boot, surv_lasso[indices][:, time_mask], times_boot))
        ibs_ds.append(integrated_brier_score(y_train_sksurv, y_test_boot, surv_ds[indices][:, time_mask], times_boot))
        ibs_dh.append(integrated_brier_score(y_train_sksurv, y_test_boot, surv_dh[indices][:, time_mask], times_boot))
        ibs_rsf.append(integrated_brier_score(y_train_sksurv, y_test_boot, surv_rsf[indices][:, time_mask], times_boot))

    # Helper function to extract Mean and 95% CI
    def get_metrics(boot_list):
        if not boot_list:
            return np.nan, (np.nan, np.nan)
        return round(np.mean(boot_list), 3), (round(np.percentile(boot_list, 2.5), 3), round(np.percentile(boot_list, 97.5), 3))

    return {
        "Dataset": dataset_name,
        # C-Index Results
        "C-Index CPH (Mean)": get_metrics(cidx_cph)[0], "C-Index CPH (95% CI)": get_metrics(cidx_cph)[1],
        "C-Index Cox Lasso (Mean)": get_metrics(cidx_lasso)[0], "C-Index Cox Lasso (95% CI)": get_metrics(cidx_lasso)[1],
        "C-Index DeepSurv (Mean)": get_metrics(cidx_ds)[0], "C-Index DeepSurv (95% CI)": get_metrics(cidx_ds)[1],
        "C-Index DeepHit (Mean)": get_metrics(cidx_dh)[0], "C-Index DeepHit (95% CI)": get_metrics(cidx_dh)[1],
        "C-Index RSF (Mean)": get_metrics(cidx_rsf)[0], "C-Index RSF (95% CI)": get_metrics(cidx_rsf)[1],

        # IBS Results
        "IBS CPH (Mean)": get_metrics(ibs_cph)[0], "IBS CPH (95% CI)": get_metrics(ibs_cph)[1],
        "IBS Cox Lasso (Mean)": get_metrics(ibs_lasso)[0], "IBS Cox Lasso (95% CI)": get_metrics(ibs_lasso)[1],
        "IBS DeepSurv (Mean)": get_metrics(ibs_ds)[0], "IBS DeepSurv (95% CI)": get_metrics(ibs_ds)[1],
        "IBS DeepHit (Mean)": get_metrics(ibs_dh)[0], "IBS DeepHit (95% CI)": get_metrics(ibs_dh)[1],
        "IBS RSF (Mean)": get_metrics(ibs_rsf)[0], "IBS RSF (95% CI)": get_metrics(ibs_rsf)[1],
    }

In [ ]:
n_bootstraps = 1000
final_results = []

final_results.append(evaluate_dataset(
    X_train_tcga, T_train_tcga, E_train_tcga,
    X_test_tcga, T_test_tcga, E_test_tcga,
    "TCGA", params_tcga, params_tcga_dh, n_bootstraps
))

df_all = pd.DataFrame(final_results)
df_cindex = df_all[['Dataset']].join(df_all.filter(like='C-Index'))
display(df_cindex)
df_ibs = df_all[['Dataset']].join(df_all.filter(like='IBS'))
display(df_ibs)